# Lab 3 — Pandas & Zomato Data Load

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

1. Load **500** Zomato rows into a pandas **DataFrame**.
2. Inspect **shape**, **dtypes**, **missing values**, and **summary statistics**.
3. Explore categories with `value_counts`, **groupby**, and **crosstab**.
4. Filter, sort, and derive columns — foundation for Seaborn (Lab 4) and regression (Labs 5–6).

> **Checkpoints:** `df.shape == (500, 9)` · mean `aggregate_rating` ≈ **3.70**

<!-- cisco-day02-lab03-expanded-2026 -->

**Dataset:** `data/zomato/zomato_restaurants.csv` — classroom sample aligned with the Kaggle Zomato use-case.

**Day 2 flow:** Labs 1–2 structures & NumPy → **Lab 3 (you are here)** pandas EDA → Lab 4 plots → Labs 5–6 linear regression.

## What is a DataFrame?

| Concept | Excel | Pandas |
|---------|-------|--------|
| Rows | Records | `index` |
| Columns | Fields | `columns` |
| Types | Number / Text | `dtypes` |
| Filters | AutoFilter | Boolean indexing / `query` |

Lab 1 gave you dicts and lists; Lab 2 gave you NumPy arrays. A DataFrame is the **labeled table** that ties both together for real analytics.

## Types of ML problems (course topic)

<!-- cisco-topic-coverage -->

| Type | Target | This training |
|------|--------|---------------|
| **Regression** | Continuous number | Predict `aggregate_rating` (Labs 5–6) |
| **Classification** | Category / yes-no | Loan default (Days 3–4) |
| **Clustering** | No labels | NYSE segments (Day 5) |
| **Anomaly detection** | Rare events | Credit card fraud (Day 6) |

Today's EDA supports **supervised** work on Zomato ratings.

---

## 1. Setup and load

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"

print("CSV path:", ZOMATO_CSV)
print("Exists:", ZOMATO_CSV.is_file())


### 1a. Peek with `nrows` before loading everything

In [ ]:
peek = pd.read_csv(ZOMATO_CSV, nrows=8)
print("Preview shape:", peek.shape)
display(peek)


In [ ]:
df = pd.read_csv(ZOMATO_CSV)

print("Lab 3 — Pandas Zomato load")
print("dataset:", ZOMATO_CSV.name)
print("shape (rows, cols):", df.shape)


In [ ]:
EXPECTED_ROWS = 500
EXPECTED_COLS = 9

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Expected ({EXPECTED_ROWS}, {EXPECTED_COLS}), got {df.shape}"
)
print("✓ Row/column count matches lab profile")


### 1b. `info()` — quick structural scan

In [ ]:
df.info()


---

## 2. Columns and dtypes

In [ ]:
print("Columns:", list(df.columns))
print()
print(df.dtypes)


| Column | Role | Dtype |
|--------|------|-------|
| `restaurant_id` | Primary key | object |
| `name` | Label | object |
| `city` | Categorical | object |
| `cuisines` | Categorical | object |
| `aggregate_rating` | **Target** (Labs 5–6) | float64 |
| `votes` | Numeric feature | int64 |
| `average_cost_for_two` | Numeric feature | int64 |
| `online_order` | Binary category | object |
| `book_table` | Binary category | object |

### 2b. Cardinality — how many distinct values?

In [ ]:
cardinality = pd.DataFrame({
    "column": df.columns,
    "nunique": [df[c].nunique() for c in df.columns],
})
display(cardinality)


---

## 3. First look — `head`, `tail`, `sample`

In [ ]:
display(df.head(5))


In [ ]:
display(df.tail(3))


In [ ]:
# Reproducible random sample (same seed as Lab 2 config)
display(df.sample(5, random_state=42))


---

## 4. Missing values and duplicates

In [ ]:
missing = df.isna().sum()
print("Missing per column:")
print(missing)
print("Any missing?", missing.any())


In [ ]:
dup_ids = df["restaurant_id"].duplicated().sum()
print("Duplicate restaurant_id rows:", dup_ids)
print("Unique IDs:", df["restaurant_id"].nunique(), "/ rows:", len(df))


---

## 5. Numeric summary — `describe`

In [ ]:
display(df.describe().round(2))


In [ ]:
mean_rating = df["aggregate_rating"].mean()
print(f"mean aggregate_rating: {mean_rating:.2f}")
assert abs(mean_rating - 3.70) < 0.05
print("✓ Mean rating checkpoint OK")


### 5b. Spread and quartiles on ratings

In [ ]:
rating = df["aggregate_rating"]
summary = {
    "min": rating.min(),
    "q1": rating.quantile(0.25),
    "median": rating.median(),
    "q3": rating.quantile(0.75),
    "max": rating.max(),
    "std": rating.std(),
}
pd.Series(summary).round(2)


### 5c. Correlation between numeric columns

In [ ]:
numeric_cols = ["aggregate_rating", "votes", "average_cost_for_two"]
display(df[numeric_cols].corr().round(3))


---

## 6. Categorical exploration

In [ ]:
print("Top cities:")
print(df["city"].value_counts().head(8))


In [ ]:
print("Online order share:")
print(df["online_order"].value_counts(normalize=True).round(3) * 100)


In [ ]:
print("Top cuisines:")
print(df["cuisines"].value_counts().head(8))


In [ ]:
print("Book table availability:")
print(df["book_table"].value_counts())


### 6b. `describe` for text columns

In [ ]:
display(df.describe(include="object").T.head(6))


### 6c. Crosstab — city × online order

In [ ]:
ct = pd.crosstab(df["city"], df["online_order"])
display(ct.head(8))


---

## 7. Selecting data — Series vs DataFrame

In [ ]:
votes_series = df["votes"]
features_df = df[["votes", "average_cost_for_two", "aggregate_rating"]]

print("Series shape:", votes_series.shape)
print("DataFrame shape:", features_df.shape)
display(features_df.head(3))


### 7b. `loc` (labels) vs `iloc` (positions)

In [ ]:
print("loc — first row by index label:")
display(df.loc[0, ["name", "city", "aggregate_rating"]])

print("iloc — first 3 rows, last 2 columns:")
display(df.iloc[:3, -2:])


---

## 8. Filtering rows

In [ ]:
highly_rated = df[df["aggregate_rating"] >= 4.5]
print(f"Rated >= 4.5: {len(highly_rated)} restaurants")

bengaluru = df[df["city"] == "Bengaluru"]
print(f"Bengaluru outlets: {len(bengaluru)}")
print("Mean rating Bengaluru:", round(bengaluru["aggregate_rating"].mean(), 2))


### 8b. `query` and `between`

In [ ]:
mid_market = df.query(
    "average_cost_for_two >= 1000 and average_cost_for_two <= 2000"
)
print("Mid-market cost band rows:", len(mid_market))

mid_ratings = df[df["aggregate_rating"].between(3.5, 4.0)]
print("Ratings between 3.5 and 4.0:", len(mid_ratings))


### 8c. Combine conditions

In [ ]:
delivery_and_top = df[
    (df["online_order"] == "Yes") & (df["aggregate_rating"] >= 4.0)
]
print("Delivery + rating >= 4.0:", len(delivery_and_top))
display(delivery_and_top[["name", "city", "aggregate_rating", "votes"]].head(5))


### 8d. `isin` — multiple cities at once

In [ ]:
metros = ["Mumbai", "Delhi", "Bengaluru"]
metro_df = df[df["city"].isin(metros)]
print("Metro subset:", len(metro_df), "of", len(df))
print("Mean rating (metros):", round(metro_df["aggregate_rating"].mean(), 2))


### 8e. `nsmallest` — budget-friendly highly-voted

In [ ]:
value_picks = df.nsmallest(5, "average_cost_for_two")
display(value_picks[["name", "city", "average_cost_for_two", "votes", "aggregate_rating"]])


---

## 9. Sorting and top performers

In [ ]:
top_rated = df.sort_values("aggregate_rating", ascending=False)
display(top_rated[["name", "city", "aggregate_rating", "votes"]].head(5))


In [ ]:
most_voted = df.nlargest(5, "votes")
display(most_voted[["name", "city", "votes", "aggregate_rating"]])


---

## 10. Derived columns

In [ ]:
eda = df.copy()
eda["cost_per_vote"] = eda["average_cost_for_two"] / eda["votes"]
eda["rating_tier"] = pd.cut(
    eda["aggregate_rating"],
    bins=[0, 3.0, 3.5, 4.0, 5.0],
    labels=["low", "mid", "good", "excellent"],
)

display(eda[["name", "aggregate_rating", "rating_tier", "cost_per_vote"]].head(6))
print("Original df still has", df.shape[1], "columns")


---

## 11. Groupby — city-level summary

In [ ]:
city_stats = (
    eda.groupby("city")
    .agg(
        outlets=("restaurant_id", "count"),
        avg_rating=("aggregate_rating", "mean"),
        avg_cost=("average_cost_for_two", "mean"),
        avg_votes=("votes", "mean"),
    )
    .round(2)
    .sort_values("avg_rating", ascending=False)
)
display(city_stats.head(10))


### 11b. Cuisine-level averages

In [ ]:
cuisine_stats = (
    eda.groupby("cuisines")["aggregate_rating"]
    .agg(["count", "mean"])
    .rename(columns={"count": "outlets", "mean": "avg_rating"})
    .round(2)
    .sort_values("outlets", ascending=False)
)
display(cuisine_stats.head(8))


### 11c. Pivot — mean rating by city and online order

In [ ]:
pivot = eda.pivot_table(
    index="city",
    columns="online_order",
    values="aggregate_rating",
    aggfunc="mean",
).round(2)
display(pivot.head(8))


### 11d. Reset index after filtering (tidy table for export)

In [ ]:
metro_top = (
    metro_df.sort_values("aggregate_rating", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(metro_top[["name", "city", "aggregate_rating"]])


---

## 12. Bridge to NumPy (Lab 2)

In [ ]:
import numpy as np

votes_np = df["votes"].to_numpy()
print("NumPy array shape:", votes_np.shape, "dtype:", votes_np.dtype)
print("Mean matches pandas:", np.isclose(votes_np.mean(), df["votes"].mean()))


---

## 13. Try it yourself

1. Filter restaurants in **Mumbai** with `book_table == 'Yes'`.
2. Compute mean `average_cost_for_two` for that subset.
3. Compare to the overall mean cost.

In [ ]:
# Your code (optional)


In [ ]:
mumbai_book = df[(df["city"] == "Mumbai") & (df["book_table"] == "Yes")]
subset_mean = mumbai_book["average_cost_for_two"].mean()
overall_mean = df["average_cost_for_two"].mean()
print("Mumbai + book_table rows:", len(mumbai_book))
print("Subset mean cost:", round(subset_mean, 0))
print("Overall mean cost:", round(overall_mean, 0))


---

## 14. Final checkpoint

In [ ]:
print("=" * 50)
print("CHECKPOINT SUMMARY")
print("=" * 50)
print(f"shape: {df.shape}")
print(f"columns ({len(df.columns)}): {list(df.columns)}")
print(f"mean aggregate_rating: {df['aggregate_rating'].mean():.2f}")
print(f"mean votes: {df['votes'].mean():.2f}")
print(f"mean average_cost_for_two: {df['average_cost_for_two'].mean():.2f}")

required = {"aggregate_rating", "votes", "average_cost_for_two", "city", "cuisines"}
assert required.issubset(df.columns)
assert df.shape == (500, 9)
print("\n✓ All checkpoint assertions passed")


---

## Reflection

1. Why is `df.shape[0]` the number of **rows**?
2. Which columns would you one-hot encode before linear regression?
3. What plots would help before modeling? *(Lab 4 — Seaborn)*
4. When is `groupby` more useful than multiple filters?

**Previous:** [Lab 2 — NumPy arrays](lab02_numpy_arrays.ipynb)  
**Next:** [Lab 4 — Seaborn plots](lab04_seaborn_plots.ipynb)